# LungCT COPD pTVPrep Visual Audit

This notebook adapts the 4DCT pTVPrep visual-audit notebook to all COPD cases 01–10. It audits the existing legacy pTV-style 1 mm preprocessing used for the controlled COPD P-family and MR-D-BSCP reference: native COPD images, legacy crop, trilinear 1 mm resampling, paired R231 lung masks, and landmark transforms.

Display convention follows `lungct_4dct_ptvprep_visual_audit.ipynb`: coronal views use radiology orientation with superior at top, inferior at bottom, patient right on image left, and patient left on image right. The yellow circles show landmarks within a thin slab around the displayed coronal plane; they are not method errors.


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from matplotlib.colors import ListedColormap


PROTOCOL_NAME = 'pTVReg_iso1mm'
RUN_PREPROCESS_IF_MISSING = False


def required_env(name: str) -> Path:
    value = os.environ.get(name)
    if not value:
        raise RuntimeError(f'Set {name} before running this notebook.')
    return Path(value)


REPO_ROOT = required_env('INRORSPLINE_REPO_ROOT')
DATA_ROOT = required_env('LUNGCT_DIRLAB_ROOT')
MASK_ROOT = DATA_ROOT / 'processed' / 'lungmask_R231_native'
PTVPREP_ROOT = DATA_ROOT / 'processed' / PROTOCOL_NAME
SUMMARY_PATH = PTVPREP_ROOT / 'COPD' / 'summary.json'
OUT_DIR = REPO_ROOT / 'outputs/data_inspection/lungct_copd_ptvprep_visual_audit'
FIGURE_DIR = OUT_DIR / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not SUMMARY_PATH.exists() and RUN_PREPROCESS_IF_MISSING:
    cmd = [
        'python',
        str(REPO_ROOT / 'scripts/data/preprocessing/lungct/generate_copd_ptvprep.py'),
        '--data-root', str(DATA_ROOT),
        '--mask-root', str(MASK_ROOT),
        '--out-root', str(PTVPREP_ROOT),
        '--case-ids', '1-10',
        '--device', 'cpu',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)

if not SUMMARY_PATH.exists():
    raise FileNotFoundError(f'Missing COPD pTVPrep summary: {SUMMARY_PATH}. Set RUN_PREPROCESS_IF_MISSING=True only if you intend to generate it.')

summary = json.loads(SUMMARY_PATH.read_text())
rows = summary['rows']
display(Markdown('**Repo:** `$INRORSPLINE_REPO_ROOT`'))
display(Markdown('**Data root:** `$LUNGCT_DIRLAB_ROOT`'))
display(Markdown('**pTVPrep root:** `$LUNGCT_DIRLAB_ROOT/processed/pTVReg_iso1mm/COPD`'))
display(Markdown('**Figure output:** `$INRORSPLINE_REPO_ROOT/outputs/data_inspection/lungct_copd_ptvprep_visual_audit/figures`'))
print('cases', len(rows), 'all_verification_pass', summary.get('all_verification_pass'))


## Load helpers and notebook-level checks

The saved preprocessing summary contains the authoritative array paths. Configure `LUNGCT_DIRLAB_ROOT` and `INRORSPLINE_REPO_ROOT` in the execution environment before opening the notebook.


In [ ]:
def resolve_saved_path(path_text: str) -> Path:
    path = Path(path_text)
    if path.exists():
        return path
    if not path.is_absolute():
        return Path(os.environ['LUNGCT_DIRLAB_ROOT']) / path
    return path


def load_case_arrays(row: dict) -> dict[str, np.ndarray]:
    return {name: np.load(resolve_saved_path(path)) for name, path in row['array_paths'].items()}


def points_inside_shape(points_zyx: np.ndarray, shape_zyx: tuple[int, int, int]) -> bool:
    pts = np.asarray(points_zyx, dtype=float)
    shape = np.asarray(shape_zyx, dtype=float)
    return bool(np.all((pts >= 0.0) & (pts <= shape[None, :] - 1.0)))


def landmark_mask_fraction(mask: np.ndarray, landmarks_zyx: np.ndarray) -> float:
    idx = np.rint(landmarks_zyx).astype(int)
    valid = (
        (idx[:, 0] >= 0) & (idx[:, 0] < mask.shape[0])
        & (idx[:, 1] >= 0) & (idx[:, 1] < mask.shape[1])
        & (idx[:, 2] >= 0) & (idx[:, 2] < mask.shape[2])
    )
    if not np.any(valid):
        return float('nan')
    return float(np.mean(mask[idx[valid, 0], idx[valid, 1], idx[valid, 2]] > 0))


audit_table = []
all_notebook_checks_pass = True

for row in rows:
    arr = load_case_arrays(row)
    fixed_crop = arr['fixed_iBH_crop_ptvintensity_zyx']
    moving_crop = arr['moving_eBH_crop_ptvintensity_zyx']
    fixed_reg = arr['fixed_iBH_reg1mm_trilinear_zyx']
    moving_reg = arr['moving_eBH_reg1mm_trilinear_zyx']
    fixed_mask_crop = arr['fixed_iBH_lungmask_crop_zyx']
    moving_mask_crop = arr['moving_eBH_lungmask_crop_zyx']
    fixed_mask_reg = arr['fixed_iBH_lungmask_reg1mm_nearest_zyx']
    moving_mask_reg = arr['moving_eBH_lungmask_reg1mm_nearest_zyx']
    fixed_lm_crop = arr['landmarks_fixed_iBH_crop_zyx']
    moving_lm_crop = arr['landmarks_moving_eBH_crop_zyx']
    fixed_lm_reg = arr['landmarks_fixed_iBH_reg1mm_zyx']
    moving_lm_reg = arr['landmarks_moving_eBH_reg1mm_zyx']

    shape_match = (
        fixed_crop.shape == fixed_mask_crop.shape
        and moving_crop.shape == moving_mask_crop.shape
        and fixed_reg.shape == fixed_mask_reg.shape
        and moving_reg.shape == moving_mask_reg.shape
    )
    crop_inside = points_inside_shape(fixed_lm_crop, fixed_crop.shape) and points_inside_shape(moving_lm_crop, moving_crop.shape)
    reg_inside = points_inside_shape(fixed_lm_reg, fixed_reg.shape) and points_inside_shape(moving_lm_reg, moving_reg.shape)
    verification_pass = bool(row['verification']['pass'])
    notebook_pass = bool(shape_match and crop_inside and reg_inside and verification_pass)
    all_notebook_checks_pass = all_notebook_checks_pass and notebook_pass

    audit_table.append({
        'case': f"{int(row['case_id']):02d}",
        'native zyx': tuple(row.get('native_shape_zyx', row['raw_shape_zyx'])),
        'crop zyx': tuple(fixed_crop.shape),
        'reg zyx': tuple(fixed_reg.shape),
        'legacy-task4 minus pTVReg': row.get('legacy_minus_ptvreg_bounds_zyx'),
        'crop LM inside': crop_inside,
        'reg LM inside': reg_inside,
        'fixed mask LM frac': f'{landmark_mask_fraction(fixed_mask_reg, fixed_lm_reg):.3f}',
        'moving mask LM frac': f'{landmark_mask_fraction(moving_mask_reg, moving_lm_reg):.3f}',
        'summary pass': verification_pass,
        'notebook pass': notebook_pass,
    })

audit_df = pd.DataFrame(audit_table)
audit_df.to_csv(OUT_DIR / 'lungct_copd_ptvprep_visual_audit_table.csv', index=False)
(OUT_DIR / 'lungct_copd_ptvprep_visual_audit_table.json').write_text(json.dumps(audit_table, indent=2) + '\n')
display(audit_df)
assert all_notebook_checks_pass


## Figure helpers

Each case uses three columns: native COPD intensity/mask, legacy pTV-style crop, and final 1 mm trilinear registration lattice. Rows are fixed `iBH` and moving `eBH`.


In [ ]:
mask_cmap_fixed = ListedColormap([(0, 0, 0, 0), (0.0, 0.85, 0.35, 0.32)])
mask_cmap_moving = ListedColormap([(0, 0, 0, 0), (0.75, 0.25, 1.0, 0.32)])


def normalize_slice(slice_2d: np.ndarray) -> np.ndarray:
    arr = np.asarray(slice_2d, dtype=np.float32)
    finite = arr[np.isfinite(arr)]
    if finite.size == 0:
        return np.zeros_like(arr)
    lo, hi = np.percentile(finite, [1, 99])
    if hi <= lo:
        lo, hi = float(finite.min()), float(finite.max())
    if hi <= lo:
        return np.zeros_like(arr)
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0)


def bbox_center_y(mask_a: np.ndarray, mask_b: np.ndarray, fallback_shape: tuple[int, int, int]) -> int:
    union = np.logical_or(mask_a > 0, mask_b > 0)
    coords = np.argwhere(union)
    if coords.size == 0:
        return int(fallback_shape[1] // 2)
    return int(round((coords[:, 1].min() + coords[:, 1].max()) / 2.0))


def draw_orientation_labels(ax) -> None:
    box = dict(facecolor='black', alpha=0.35, pad=1, edgecolor='none')
    ax.text(0.50, 0.985, 'Superior', ha='center', va='top', color='white', fontsize=7, transform=ax.transAxes, bbox=box)
    ax.text(0.50, 0.015, 'Inferior', ha='center', va='bottom', color='white', fontsize=7, transform=ax.transAxes, bbox=box)
    ax.text(0.015, 0.50, 'Patient R', ha='left', va='center', color='white', fontsize=7, rotation=90, transform=ax.transAxes, bbox=box)
    ax.text(0.985, 0.50, 'Patient L', ha='right', va='center', color='white', fontsize=7, rotation=90, transform=ax.transAxes, bbox=box)


def draw_coronal_panel(ax, volume, mask, landmarks_zyx, y_idx, spacing_zyx, title, cmap, slab) -> None:
    y_idx = int(np.clip(y_idx, 0, volume.shape[1] - 1))
    img = np.flip(volume[:, y_idx, :], axis=0)
    msk = np.flip((mask[:, y_idx, :] > 0).astype(np.uint8), axis=0)
    aspect = float(spacing_zyx[0]) / float(spacing_zyx[2])
    ax.imshow(normalize_slice(img), cmap='gray', origin='lower', aspect=aspect, interpolation='nearest')
    ax.imshow(np.ma.masked_where(msk == 0, msk), cmap=cmap, origin='lower', aspect=aspect, interpolation='nearest', vmin=0, vmax=1)
    pts = np.asarray(landmarks_zyx, dtype=float)
    near = np.abs(pts[:, 1] - y_idx) <= slab
    if np.any(near):
        ax.scatter(pts[near, 2], (volume.shape[0] - 1) - pts[near, 0], s=10, facecolors='none', edgecolors='yellow', linewidths=0.5, alpha=0.8)
    ax.set_title(f'{title}\ny={y_idx}; yellow = landmarks within ±{slab:g} voxels', fontsize=9)
    ax.axis('off')
    draw_orientation_labels(ax)


## All-case figure review

These panels are meant to catch gross preprocessing issues: phase swap, crop truncation, mask/crop mismatch, severe field-of-view mismatch, or display-orientation confusion. They are not registration results.


In [ ]:
saved_figures = []

for row in rows:
    arr = load_case_arrays(row)
    cid = int(row['case_id'])
    native_spacing = tuple(float(v) for v in row['native_spacing_zyx_mm'])
    reg_spacing = (1.0, 1.0, 1.0)

    fixed_native = arr['fixed_iBH_native_ptvintensity_zyx']
    moving_native = arr['moving_eBH_native_ptvintensity_zyx']
    fixed_crop = arr['fixed_iBH_crop_ptvintensity_zyx']
    moving_crop = arr['moving_eBH_crop_ptvintensity_zyx']
    fixed_reg = arr['fixed_iBH_reg1mm_trilinear_zyx']
    moving_reg = arr['moving_eBH_reg1mm_trilinear_zyx']
    fixed_mask_native = arr['fixed_iBH_lungmask_native_zyx']
    moving_mask_native = arr['moving_eBH_lungmask_native_zyx']
    fixed_mask_crop = arr['fixed_iBH_lungmask_crop_zyx']
    moving_mask_crop = arr['moving_eBH_lungmask_crop_zyx']
    fixed_mask_reg = arr['fixed_iBH_lungmask_reg1mm_nearest_zyx']
    moving_mask_reg = arr['moving_eBH_lungmask_reg1mm_nearest_zyx']
    fixed_lm_native = arr['landmarks_fixed_iBH_native0_zyx']
    moving_lm_native = arr['landmarks_moving_eBH_native0_zyx']
    fixed_lm_crop = arr['landmarks_fixed_iBH_crop_zyx']
    moving_lm_crop = arr['landmarks_moving_eBH_crop_zyx']
    fixed_lm_reg = arr['landmarks_fixed_iBH_reg1mm_zyx']
    moving_lm_reg = arr['landmarks_moving_eBH_reg1mm_zyx']

    y_native = bbox_center_y(fixed_mask_native, moving_mask_native, fixed_native.shape)
    y_crop = bbox_center_y(fixed_mask_crop, moving_mask_crop, fixed_crop.shape)
    y_reg = bbox_center_y(fixed_mask_reg, moving_mask_reg, fixed_reg.shape)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), constrained_layout=True)
    fig.suptitle(
        f'COPD case {cid:02d} | legacy pTV-style crop + trilinear 1 mm | '
        f'crop {tuple(row["crop_shape_zyx"])} -> reg {tuple(row["registration_shape_zyx"])}',
        fontsize=11,
    )
    draw_coronal_panel(axes[0, 0], fixed_native, fixed_mask_native, fixed_lm_native, y_native, native_spacing, 'iBH fixed native + mask', mask_cmap_fixed, 4.0)
    draw_coronal_panel(axes[1, 0], moving_native, moving_mask_native, moving_lm_native, y_native, native_spacing, 'eBH moving native + mask', mask_cmap_moving, 4.0)
    draw_coronal_panel(axes[0, 1], fixed_crop, fixed_mask_crop, fixed_lm_crop, y_crop, native_spacing, 'iBH fixed legacy crop', mask_cmap_fixed, 4.0)
    draw_coronal_panel(axes[1, 1], moving_crop, moving_mask_crop, moving_lm_crop, y_crop, native_spacing, 'eBH moving legacy crop', mask_cmap_moving, 4.0)
    draw_coronal_panel(axes[0, 2], fixed_reg, fixed_mask_reg, fixed_lm_reg, y_reg, reg_spacing, 'iBH fixed 1 mm reg lattice', mask_cmap_fixed, 5.0)
    draw_coronal_panel(axes[1, 2], moving_reg, moving_mask_reg, moving_lm_reg, y_reg, reg_spacing, 'eBH moving 1 mm reg lattice', mask_cmap_moving, 5.0)

    fig_path = FIGURE_DIR / f'copd_case{cid:02d}_ptvprep_visual_audit.png'
    fig.savefig(fig_path, dpi=160)
    saved_figures.append({'case': f'{cid:02d}', 'figure': str(fig_path.relative_to(REPO_ROOT)), 'y_native': int(y_native), 'y_crop': int(y_crop), 'y_reg': int(y_reg)})
    display(Markdown(f'### COPD case {cid:02d}'))
    display(Image(filename=str(fig_path), width=1100))
    plt.close(fig)

fig_df = pd.DataFrame(saved_figures)
fig_df.to_csv(OUT_DIR / 'lungct_copd_ptvprep_visual_audit_figures.csv', index=False)
(OUT_DIR / 'lungct_copd_ptvprep_visual_audit_figures.json').write_text(json.dumps(saved_figures, indent=2) + '\n')
display(fig_df)
